# 环节 05 · FFN / 激活 / MoE（配套 Notebook）

> 配套长文：[环节05-FFN激活与MoE详解.md](./环节05-FFN激活与MoE详解.md)
> 定位：把"FFN 是参数大头""SwiGLU 门控""MoE 大参数小激活"三件事**算成数**。纯 Python 标准库，零依赖。

**怎么跑**：逐格 `Shift+Enter`；后面的格子依赖前面已执行的变量。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 FFN 参数账 | §1 / §2 | LLaMA-2-7B 里 FFN 占每层参数的 67% |
| §2 三种激活 | §3 | ReLU 死区 / GELU 平滑 / SiLU 有负值 |
| §3 SwiGLU 门控 | §3 | "阀门"如何决定放行多少 |
| §4 MoE 路由与均衡 | §4 | Top-K 路由、专家使用不均、aux-loss |
| §5 稠密 vs MoE | §4.3 | 总参数 671B / 激活 37B 是什么概念 |


## 1. FFN 是参数大头（长文 §1 / §2）

`FFN(x) = W_down · σ(W_up · x)`：升维（d → ~4d）→ 激活 → 降维。SwiGLU 变体是三组矩阵（`W_gate / W_up / W_down`），因此 `d_ff` 取 `8/3·d` 以保持参数量与标准版相当。

拿 LLaMA-2-7B 的真实形状算一遍，看 FFN 到底占多少：


In [ ]:
d = 4096          # hidden
L = 32            # 层数
d_ff = 11008      # LLaMA-2-7B 的 FFN 中间维度
V = 32000         # 词表

attn_per_layer = 4 * d * d          # W_Q / W_K / W_V / W_O 各 d×d
ffn_per_layer = 3 * d * d_ff        # SwiGLU 三矩阵
layer_total = attn_per_layer + ffn_per_layer

print(f"LLaMA-2-7B 形状：d={d}, L={L}, d_ff={d_ff}, V={V}\n")
print(f"  每层 Attention 4d²     = {attn_per_layer/1e6:>7.1f} M 参数")
print(f"  每层 FFN     3·d·d_ff  = {ffn_per_layer/1e6:>7.1f} M 参数"
      f"   ← 占每层的 {ffn_per_layer/layer_total:.1%}")
print(f"  {L} 层合计              = {layer_total*L/1e9:>7.2f} B")
print(f"  + Embedding {V}×{d}     = {V*d/1e6:>7.1f} M")
print(f"  总计 ≈ {(layer_total*L + V*d)/1e9:.2f} B（即常说的 7B）")

print(f"\n注：8/3·d = {8*d/3:.0f}，实际取 {d_ff}（对齐到 256 的整数倍，工程习惯）")
print("→ 长文说 FFN“约占 2/3”，这里算出来是 66.8%，对上了。")


## 2. 三种激活函数（长文 §3）

`ReLU → GELU → SwiGLU` 的演进动机：**从"一刀切"到"平滑"再到"可控放行"**。


In [ ]:
import math


def relu(x):
    return max(0.0, x)


def gelu(x):
    return 0.5 * x * (1 + math.erf(x / math.sqrt(2)))


def silu(x):
    return x / (1 + math.exp(-x))          # = x·σ(x)，SwiGLU 用的就是它


xs = [-3, -2, -1, -0.5, 0, 0.5, 1, 2, 3]
print(f"{'x':>6} {'ReLU':>9} {'GELU':>9} {'SiLU':>9}")
print("-" * 36)
for x in xs:
    print(f"{x:>6} {relu(x):>9.4f} {gelu(x):>9.4f} {silu(x):>9.4f}")

print("\n→ ReLU 在负半轴恒为 0（死区，梯度也是 0，一票否决）")
print("  GELU 光滑，负值处保留一点小梯度（深层训练更稳）")
print("  SiLU 在 x<0 时输出**负值**——负信息没被丢掉，只是被缩放了")


## 3. SwiGLU 的"阀门"机制（长文 §3）

```
FFN(x) = ( SiLU(x·W_gate) ⊙ (x·W_up) ) · W_down
          └─ 阀门：0~1 之间 ─┘   └─ 内容 ─┘
```

`W_gate` 一路负责**决定放行多少**，`W_up` 一路负责**提供内容**，逐元素相乘。这是它与 ReLU/GELU 的本质差别——激活不再是"对内容做非线性"，而是"用一个学出来的门去控制另一条路"。


In [ ]:
import random

random.seed(0)
dg = 4
x = [0.5, -1.2, 2.0, 0.3]
W_gate = [random.gauss(0, 0.5) for _ in range(dg)]
W_up = [random.gauss(0, 0.5) for _ in range(dg)]

gate_pre = sum(a * b for a, b in zip(x, W_gate))
up_pre = sum(a * b for a, b in zip(x, W_up))
gate = silu(gate_pre)

print(f"输入 x            = {x}")
print(f"gate 支路 W_gate·x = {gate_pre:+.4f}")
print(f"  → SiLU 后阀门开度 = {gate:+.4f}")
print(f"up   支路 W_up·x   = {up_pre:+.4f}   （内容）")
print(f"输出 = 阀门 × 内容 = {gate * up_pre:+.4f}")

print("\n换成 ReLU 当门：")
print(f"  ReLU({gate_pre:+.4f}) = {relu(gate_pre):.4f}"
      f" → 输出 {relu(gate_pre) * up_pre:+.4f}")
print(f"  而 SiLU 门给出 {gate:+.4f} → 输出 {gate * up_pre:+.4f}")
print("\n→ 关键差别：gate_pre 落在负区间时，ReLU 直接把这条通路**关死**（梯度也断）；")
print("  SiLU 保留一个小负值，通路仍能回传梯度。这就是“门控比激活表达力强”的物理含义。")


## 4. MoE：Router + Top-K + 负载均衡（长文 §4）

MoE 层 = K 个"专家 FFN" + 1 个 Router。Router 给每个 token 对每个专家打分，取 Top-K 加权混合。


In [ ]:
random.seed(1)
E, K = 8, 2                                    # 8 个专家，每 token 选 2 个
router_W = [[random.gauss(0, 0.5) for _ in range(4)] for _ in range(E)]


def route(tok):
    """返回 (Top-K 专家, 归一化门控权重, 全专家概率)。"""
    scores = [sum(a * b for a, b in zip(tok, router_W[e])) for e in range(E)]
    m = max(scores)
    exps = [math.exp(s - m) for s in scores]
    z = sum(exps)
    probs = [e / z for e in exps]
    top = sorted(range(E), key=lambda e: -probs[e])[:K]
    zw = sum(probs[e] for e in top)
    return top, [probs[e] / zw for e in top], probs


tok = [0.8, -0.3, 1.1, 0.2]
top, gw, probs = route(tok)
print(f"一个 token 的路由：")
print(f"  各专家打分概率  {[round(p, 4) for p in probs]}")
print(f"  Top-{K} 专家      {top}   门控权重 {[round(g, 4) for g in gw]}")
print(f"  最终输出 = {gw[0]:.3f}·专家{top[0]}(x) + {gw[1]:.3f}·专家{top[1]}(x)")
print(f"  （其余 {E - K} 个专家这一次**完全不参与计算** —— 这就是“小激活”）")


In [ ]:
# 批量统计：专家使用是否均衡？为什么需要负载均衡？
random.seed(1)
router_W = [[random.gauss(0, 0.5) for _ in range(4)] for _ in range(E)]
tokens = [[random.gauss(0, 1) for _ in range(4)] for _ in range(200)]

counts = [0] * E
gate_sum = [0.0] * E
for tok in tokens:
    top, _, probs = route(tok)
    for e in top:
        counts[e] += 1
    for e in range(E):
        gate_sum[e] += probs[e]

f = [cnt / (len(tokens) * K) for cnt in counts]      # 分给专家 i 的 token 比例
P = [g / len(tokens) for g in gate_sum]              # 专家 i 的平均路由概率
aux_loss = E * sum(fi * pi for fi, pi in zip(f, P))  # Switch Transformer 的负载均衡损失

print(f"专家使用次数：{counts}")
print(f"最热 : 最冷 = {max(counts)} : {min(counts)}")
print(f"“专家坍缩”程度：最热的专家拿走了 {max(counts)/sum(counts):.1%} 的分配")
print(f"\n负载均衡 aux-loss = {aux_loss:.4f}")
print(f"  完全均匀时的理想值 = K/E = {K/E:.4f}")
print(f"  → 越不均匀越大；训练时会把它加进总损失一起优化")

perfect = [len(tokens) * K / E] * E
print(f"\n对照：如果完全均匀（每个专家各 {perfect[0]:.0f} 次），aux-loss 会是 {K/E:.4f}")


## 5. 稠密 vs MoE：同一句话的两个数字（长文 §4.3）

**总参数**决定显存（所有专家必须驻留），**激活参数**决定算力（每 token 只算 Top-K）。


In [ ]:
print(f"{'方案':<20} {'总参数':>8} {'每 token 激活':>14} {'激活占比':>10} {'显存':>12}")
print("-" * 70)
for name, total_b, act_b, gb_per_b in [
    ("稠密 7B (BF16)", 7, 7, 2.0),
    ("稠密 70B (BF16)", 70, 70, 2.0),
    ("DeepSeek-V3 (MoE)", 671, 37, 1.0),        # 671B 里含 MLA 等，量级示意
]:
    print(f"{name:<20} {total_b:>6} B {act_b:>12} B {act_b/total_b:>9.0%}"
          f" {total_b*gb_per_b:>10.0f} GB")

print("\n读法（长文 §4.3 口诀）：MoE = 用显存换“大容量 + 小激活”")
print("  · 显存看总参数：671B 必须全部驻留（所以要专家并行 EP 摊到多卡）")
print("  · 算力看激活参数：每 token 只算 37B → 单 token 推理成本接近一个 37B 稠密模型")
print("  · 再加 MLA（KV 极小）+ 引擎调度，才有了 DeepSeek 671B 的低成本推理")
print("\n代价（长文 §4.4）：小显存单卡跑不动；路由抖动影响批处理效率；")
print("负载不均要靠 aux-loss / 共享专家 / 无辅助损失路由（DeepSeek 路线）来治。")


## 6. 自测（长文 §5）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| FFN 和 Attention 谁参数多？ | §1：每层 FFN 占 66.8%，全模型参数大头 |
| SwiGLU 凭什么更好？ | §3：门控决定放行量；ReLU 会把负区间通路关死 |
| d_ff 为什么是 8/3·d？ | §1：三组矩阵，取 8/3 才与标准 FFN 参数量持平 |
| MoE 的"大参数小激活"是什么账？ | §5：总参数管显存、激活参数管算力 |
| 专家坍缩怎么防？ | §4：最热专家独吞 1/4 的分配 → 用 aux-loss 压回均匀 |

**下一站**：[环节 06 · 残差与归一化](./环节06-残差连接与归一化详解.md)——FFN 和 Attention 外面那对"脚手架"。
